# Function Calling Anthropic Agent

### This notebook shows you how to use our Anthropic agent, powered by function calling capabilities.



In [2]:
# Import required libraries
from llama_index.core.tools import FunctionTool
from llama_index.llms.anthropic import Anthropic
from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.agent import FunctionCallingAgentWorker
import os
from dotenv import load_dotenv
from ragaai_catalyst.tracers import Tracer
from ragaai_catalyst import RagaAICatalyst, init_tracing
from ragaai_catalyst import trace_tool, trace_llm, trace_agent
import nest_asyncio

# Initialize nest_asyncio
nest_asyncio.apply()

# Load environment variables
from dotenv import load_dotenv

load_dotenv()

# Verify environment variables
required_env_vars = [
    "RAGAAI_CATALYST_ACCESS_KEY",
    "RAGAAI_CATALYST_SECRET_KEY",
    "RAGAAI_CATALYST_BASE_URL",
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY"
]

missing_vars = [var for var in required_env_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

# Initialize RagaAI Catalyst
try:
    catalyst = RagaAICatalyst(
        access_key=os.getenv("RAGAAI_CATALYST_ACCESS_KEY"),
        secret_key=os.getenv("RAGAAI_CATALYST_SECRET_KEY"),
        base_url=os.getenv("RAGAAI_CATALYST_BASE_URL"),
    )
    print("RagaAI Catalyst initialized successfully")
except Exception as e:
    raise Exception(f"Failed to initialize RagaAI Catalyst: {str(e)}")

# Initialize tracer
tracer = Tracer(
    project_name="anthropic_agent",
    dataset_name="anthropic_agent",
    tracer_type="Agentic",
)

# Initialize tracing
try:
    init_tracing(catalyst=catalyst, tracer=tracer)
    print("Tracing initialized successfully")
except Exception as e:
    raise Exception(f"Error initializing tracing: {str(e)}")

@trace_tool(name="multiply_tool")
def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b

multiply_tool = FunctionTool.from_defaults(fn=multiply)

@trace_tool(name="add_tool")
def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b

add_tool = FunctionTool.from_defaults(fn=add)

# Initialize Anthropic LLM with tracing
@trace_llm(name="initialize_llm")
def initialize_llm():
    return Anthropic(
        model="claude-3-sonnet-20240229",
        api_key=os.getenv("ANTHROPIC_API_KEY")
    )

llm = initialize_llm()

# Initialize agent worker with tracing
@trace_agent(name="initialize_agent")
def initialize_agent(tools, llm, allow_parallel=False):
    return FunctionCallingAgentWorker.from_tools(
        tools=tools,
        llm=llm,
        verbose=True,
        allow_parallel_tool_calls=allow_parallel
    )

# Initialize document processing with tracing
@trace_tool(name="process_documents")
def process_documents(file_path):
    return SimpleDirectoryReader(input_files=[file_path]).load_data()

# Initialize vector store with tracing
@trace_tool(name="initialize_vector_store")
def initialize_vector_store(documents, embed_model):
    return VectorStoreIndex.from_documents(documents, embed_model=embed_model)

# Main execution
try:
    with tracer:
        # Initialize basic agent
        agent_worker = initialize_agent([multiply_tool, add_tool], llm)
        agent = agent_worker.as_agent()
        print("Basic agent initialized successfully")

        # Test basic arithmetic operations
        print("\nTesting basic arithmetic operations:")
        response = agent.chat("What is (121 + 2) * 5?")
        print(f"Response: {str(response)}")
        print(f"Sources: {response.sources}")

        # Initialize parallel agent
        parallel_agent_worker = initialize_agent([multiply_tool, add_tool], llm, True)
        parallel_agent = parallel_agent_worker.as_agent()
        print("\nParallel agent initialized successfully")

        # Test parallel operations
        print("\nTesting parallel operations:")
        response = await parallel_agent.achat("What is (121 * 3) + (5 * 8)?")
        print(f"Response: {str(response)}")

        # Initialize document processing
        print("\nInitializing document processing:")
        
        # Create directory if it doesn't exist
        os.makedirs('data/10k', exist_ok=True)
        
        # Download and process Uber 10K document
        url = 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/10k/uber_2021.pdf'
        file_path = 'data/10k/uber_2021.pdf'
        
        # Download file if it doesn't exist
        if not os.path.exists(file_path):
            import requests
            response = requests.get(url)
            with open(file_path, 'wb') as file:
                file.write(response.content)
            print(f"File downloaded to {file_path}")

        # Initialize embedding model
        embed_model = OpenAIEmbedding(api_key=os.getenv("OPENAI_API_KEY"))
        query_llm = Anthropic(
            model="claude-3-haiku-20240307",
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )

        # Process documents and initialize vector store
        uber_docs = process_documents(file_path)
        uber_index = initialize_vector_store(uber_docs, embed_model)
        
        # Create query engine and tool
        uber_engine = uber_index.as_query_engine(similarity_top_k=3, llm=query_llm)
        query_engine_tool = QueryEngineTool(
            query_engine=uber_engine,
            metadata=ToolMetadata(
                name="uber_10k",
                description="Provides information about Uber financials for year 2021. Use a detailed plain text question as input."
            ),
        )

        # Initialize document agent
        doc_agent_worker = initialize_agent([query_engine_tool], llm)
        doc_agent = doc_agent_worker.as_agent()
        print("\nDocument agent initialized successfully")

        # Test document query
        print("\nTesting document query:")
        response = doc_agent.chat("Tell me both the risk factors and tailwinds for Uber?")
        print(f"Response: {str(response)}")

        print("\nAll tests completed successfully")

except Exception as e:
    print(f"Error during execution: {str(e)}")
    raise

finally:
    print("\nClosing tracer...")
    tracer.stop()

Token(s) set successfully
RagaAI Catalyst initialized successfully
Tracing initialized successfully
Basic agent initialized successfully

Testing basic arithmetic operations:
Added user message to memory: What is (121 + 2) * 5?


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


=== LLM Response ===
Okay, let's break this down step-by-step:
=== Calling Function ===
Calling function: add with args: {"a": 121, "b": 2}
=== Function Output ===
123


INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


=== LLM Response ===
First, we add 121 and 2 to get 123.

Then:
=== Calling Function ===
Calling function: multiply with args: {"a": 123, "b": 5}
=== Function Output ===
615


...

=== Function Output ===
Based on the provided context, the major risk factors for Uber in 2021 included:

1. Concentration of business in large metropolitan areas and trips to/from airports: Uber derived a significant portion of its Mobility Gross Bookings from a few large metropolitan areas and trips to/from airports. This made Uber's business susceptible to economic, social, weather, and regulatory conditions in these key areas.

2. Regulatory challenges: Uber faced regulatory obstacles in some markets, such as New York City's regulations on vehicle licenses and minimum pay standards for drivers, which adversely impacted Uber's financial performance.

3. Impacts of the COVID-19 pandemic: The pandemic led to changes in travel behavior and reduced demand for Uber's Mobility offerings, as well as driver supply constraints that continued to impact the business.

On the positive side, the context highlights that Uber's Delivery segment expanded the pool of drivers and provided benefits to

INFO:httpx:HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
INFO:ragaai_catalyst.tracers.agentic_tracing.utils.zip_list_of_unique_files: Zip file created successfully.


=== LLM Response ===
The key risk factors mentioned include Uber's geographic concentration, regulatory hurdles in certain markets, and the ongoing impacts from COVID-19 on mobility demand and driver supply. A positive tailwind was the growth of Uber's Delivery segment, which expanded their driver pool and provided value to customers and merchants.
Response: The key risk factors mentioned include Uber's geographic concentration, regulatory hurdles in certain markets, and the ongoing impacts from COVID-19 on mobility demand and driver supply. A positive tailwind was the growth of Uber's Delivery segment, which expanded their driver pool and provided value to customers and merchants.

All tests completed successfully


INFO:ragaai_catalyst.tracers.agentic_tracing.tracers.base: Traces saved successfully.


Uploading agentic traces...
Uploading code...
Dataset trace code inserted successfully

Closing tracer...
